In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.window import Window

In [2]:
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.12.1"
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin")
    + os.pathsep
    + os.environ["PATH"]
)

# SOAL 1 — Data Loading & Profiling Awal

## Perintah 1:
Load keempat CSV menjadi DataFrame Spark dengan schema yang benar (pastikan order_date dan signup_date bertipe date, total_amount, price double, quantity int).

In [3]:
spark = SparkSession.builder \
    .appName("e-commerce_growth_and_profitability_analytics") \
    .master("local[*]") \
    .getOrCreate()

c:\Users\acer-\miniconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
df_orders = spark.read \
               .option("header", "true") \
               .option("inferSchema", "true") \
               .csv("datasets/orders.csv")

# Mengecek skema DataFrame orders
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)



In [5]:
df_order_items = spark.read \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .csv("datasets/order_items.csv")

# mengecek skema DataFrame order_items
df_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



In [6]:
df_products = spark.read \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .csv("datasets/products.csv")

# mengecek skema DataFrame products
df_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [7]:
df_users = spark.read \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .csv("datasets/users.csv")
# mengecek skema DataFrame users
df_users.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)



## Perintah 2:
Lakukan data profiling ringkas:
1. jumlah baris tiap DataFrame,
2. cek null per kolom,
3. tampilkan 5 contoh baris tiap dataset.

In [8]:
# Jumlah baris tiap DataFrame
count_df_orders = df_orders.count()
count_df_order_items = df_order_items.count()
count_df_products = df_products.count()
count_df_users = df_users.count()
print("Jumlah baris tiap DataFrame:")
print(f"DataFrame Orders: {count_df_orders}")
print(f"DataFrame Order Items: {count_df_order_items}")
print(f"DataFrame Products: {count_df_products}")
print(f"DataFrame Users: {count_df_users}")

Jumlah baris tiap DataFrame:
DataFrame Orders: 18
DataFrame Order Items: 20
DataFrame Products: 10
DataFrame Users: 14


In [9]:
# Cek null pada setiap DataFrame
null_counts_orders = df_orders.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_orders.columns])
null_counts_order_items = df_order_items.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_order_items.columns])
null_counts_products = df_products.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_products.columns])
null_counts_users = df_users.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_users.columns])
print("Jumlah nilai null pada setiap DataFrame:")
print("DataFrame Orders:")
null_counts_orders.show()
print("DataFrame Order Items:")
null_counts_order_items.show()
print("DataFrame Products:")
null_counts_products.show() 
print("DataFrame Users:")
null_counts_users.show()

Jumlah nilai null pada setiap DataFrame:
DataFrame Orders:
+--------+-------+----------+------------+
|order_id|user_id|order_date|total_amount|
+--------+-------+----------+------------+
|       1|      1|         0|           0|
+--------+-------+----------+------------+

DataFrame Order Items:
+--------+----------+--------+-----+
|order_id|product_id|quantity|price|
+--------+----------+--------+-----+
|       1|         1|       0|    0|
+--------+----------+--------+-----+

DataFrame Products:
+----------+------------+--------+-----+
|product_id|product_name|category|price|
+----------+------------+--------+-----+
|         1|           0|       0|    0|
+----------+------------+--------+-----+

DataFrame Users:
+-------+----+----+-----------+
|user_id|name|city|signup_date|
+-------+----+----+-----------+
|      1|   0|   0|          0|
+-------+----+----+-----------+



In [10]:
# Menampilkan 5 contoh baris dari setiap DataFrame
print("Contoh baris dari DataFrame Orders:")
df_order_items.show(5)
print("Contoh baris dari DataFrame Order Items:")
df_order_items.show(5)
print("Contoh baris dari DataFrame Products:")
df_products.show(5)
print("Contoh baris dari DataFrame Users:")
df_users.show(5)

Contoh baris dari DataFrame Orders:
+--------+----------+--------+---------+
|order_id|product_id|quantity|    price|
+--------+----------+--------+---------+
|    O001|      P001|       1|3500000.0|
|    O001|      P001|       1|3500000.0|
|    O002|      P008|       2| 200000.0|
|    O003|      P002|       1|8500000.0|
|    O004|      P003|       1| 150000.0|
+--------+----------+--------+---------+
only showing top 5 rows
Contoh baris dari DataFrame Order Items:
+--------+----------+--------+---------+
|order_id|product_id|quantity|    price|
+--------+----------+--------+---------+
|    O001|      P001|       1|3500000.0|
|    O001|      P001|       1|3500000.0|
|    O002|      P008|       2| 200000.0|
|    O003|      P002|       1|8500000.0|
|    O004|      P003|       1| 150000.0|
+--------+----------+--------+---------+
only showing top 5 rows
Contoh baris dari DataFrame Products:
+----------+-------------+------------+---------+
|product_id| product_name|    category|    price|

## Perintah 3:
Buat “anomaly summary” awal:
1. jumlah orders dengan total_amount <= 0,
2. jumlah order_items dengan quantity <= 0 atau price <= 0,
3. jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users).

In [11]:
# anomaly summary jumlah orders dengan total_amount <= 0
anomaly_summary_orders  = df_orders.filter(df_orders.total_amount <= 0)
print("Jumlah orders dengan total_amount <= 0:", anomaly_summary_orders.count())
print("Data orders dengan total_amount <= 0:")
anomaly_summary_orders.show(5)

Jumlah orders dengan total_amount <= 0: 2
Data orders dengan total_amount <= 0:
+--------+-------+----------+------------+
|order_id|user_id|order_date|total_amount|
+--------+-------+----------+------------+
|    O007|   U006|2023-10-06|    -50000.0|
|    O008|   U007|2023-10-07|         0.0|
+--------+-------+----------+------------+



In [12]:
# anomaly summary jumlah order_items dengan quantity <= 0 atau price <= 0
anomaly_summary_order_items = df_order_items.filter((df_order_items.quantity <= 0) | (df_order_items.price <= 0))
print("Jumlah order_items dengan quantity <= 0 atau price <= 0:", anomaly_summary_order_items.count())
print("Data order_items dengan quantity <= 0 atau price <= 0:")
anomaly_summary_order_items.show(5)

Jumlah order_items dengan quantity <= 0 atau price <= 0: 2
Data order_items dengan quantity <= 0 atau price <= 0:
+--------+----------+--------+--------+
|order_id|product_id|quantity|   price|
+--------+----------+--------+--------+
|    O007|      P003|      -1|150000.0|
|    O008|      P004|       1|-50000.0|
+--------+----------+--------+--------+



In [13]:
# anomaly summary jumlah duplikasi key utama (order_id di orders, product_id di products, user_id di users)
anomaly_summary_duplicates_orders = df_orders.groupBy("order_id").count().filter("count > 1")
print("Jumlah duplikasi order_id:", anomaly_summary_duplicates_orders.count())
print("Data duplikasi order_id:")
anomaly_summary_duplicates_orders.show(5)

anomaly_summary_duplicates_products = df_products.groupBy("product_id").count().filter("count > 1")
print("Jumlah duplikasi product_id:", anomaly_summary_duplicates_products.count())
print("Data duplikasi product_id:")
anomaly_summary_duplicates_products.show(5)

anomaly_summary_duplicates_users = df_users.groupBy("user_id").count().filter("count > 1")
print("Jumlah duplikasi user_id:", anomaly_summary_duplicates_users.count())
print("Data duplikasi user_id:")
anomaly_summary_duplicates_users.show(5)

Jumlah duplikasi order_id: 1
Data duplikasi order_id:
+--------+-----+
|order_id|count|
+--------+-----+
|    O003|    2|
+--------+-----+

Jumlah duplikasi product_id: 1
Data duplikasi product_id:
+----------+-----+
|product_id|count|
+----------+-----+
|      P001|    2|
+----------+-----+

Jumlah duplikasi user_id: 1
Data duplikasi user_id:
+-------+-----+
|user_id|count|
+-------+-----+
|   U002|    2|
+-------+-----+



# SOAL 2 — Cleaning & Standardization

## Perintah 1:
Bersihkan data:
1. drop baris orders yang order_id atau user_id null,
2. drop baris order_items yang order_id atau product_id null,
3. drop baris products yang product_id null,
4. drop baris users yang user_id null.

In [14]:
# cleaning orders
df_orders_cleaned = df_orders.filter((df_orders.order_id.isNotNull()) & (df_orders.user_id.isNotNull()))
print("Jumlah baris sebelum cleaning orders:", df_orders.count())
print("Jumlah baris setelah cleaning orders:", df_orders_cleaned.count())

Jumlah baris sebelum cleaning orders: 18
Jumlah baris setelah cleaning orders: 16


In [15]:
# cleaning order_items
df_order_items_cleaned = df_order_items.filter((df_order_items.order_id.isNotNull()) & (df_order_items.product_id.isNotNull()))
print("Jumlah baris sebelum cleaning order_items:", df_order_items.count())
print("Jumlah baris setelah cleaning order_items:", df_order_items_cleaned.count())

Jumlah baris sebelum cleaning order_items: 20
Jumlah baris setelah cleaning order_items: 18


In [16]:
# cleaning products
df_products_cleaned = df_products.filter(df_products.product_id.isNotNull())
print("Jumlah baris sebelum cleaning products:", df_products.count())
print("Jumlah baris setelah cleaning products:", df_products_cleaned.count())

Jumlah baris sebelum cleaning products: 10
Jumlah baris setelah cleaning products: 9


In [17]:
# cleaning users
df_users_cleaned = df_users.filter(df_users.user_id.isNotNull())
print("Jumlah baris sebelum cleaning users:", df_users.count())
print("Jumlah baris setelah cleaning users:", df_users_cleaned.count())

Jumlah baris sebelum cleaning users: 14
Jumlah baris setelah cleaning users: 13


## Perintah 2:
Standardisasi kolom teks:
1. city di users → lowercase + trim,
2. category di products → lowercase + trim.

In [18]:
# standardize city in users DataFrame (lowercase and trim)
df_users_standardized = df_users_cleaned.withColumn("city", F.lower(F.col("city"))) \
                                        .withColumn("city", F.trim(F.col("city")))

df_users_standardized.show(5)


+-------+-------------+--------+-----------+
|user_id|         name|    city|signup_date|
+-------+-------------+--------+-----------+
|   U001| Andi Pratama| jakarta| 2023-01-15|
|   U002| Budi Santoso|surabaya| 2023-03-20|
|   U002| Budi Santoso|surabaya| 2023-03-20|
|   U003|   Citra Dewi| bandung| 2023-06-10|
|   U004|Dimas Anggara| jakarta| 2023-07-01|
+-------+-------------+--------+-----------+
only showing top 5 rows


In [19]:
# standardize category in products DataFrame (lowercase and trim)
df_products_standardized = df_products_cleaned.withColumn("category", F.lower(F.col("category"))) \
                                        .withColumn("category", F.trim(F.col("category")))
df_products_standardized.show(5)

+----------+-------------+----------+---------+
|product_id| product_name|  category|    price|
+----------+-------------+----------+---------+
|      P001| Smartphone X|elektronik|3500000.0|
|      P001| Smartphone X|elektronik|3500000.0|
|      P002|   Laptop Pro|elektronik|8500000.0|
|      P003|Kemeja Casual|   fashion| 150000.0|
|      P004| Celana Jeans|   fashion| 250000.0|
+----------+-------------+----------+---------+
only showing top 5 rows


## Perintah 3:
Deduplication berbasis key:
1. orders dedupe by order_id,
2. order_items dedupe by kombinasi (order_id,product_id),
3. products dedupe by product_id,
4. users dedupe by user_id.
<p>Tunjukkan jumlah baris sebelum vs sesudah dedupe.</p>

In [20]:
# deduplicate orders by order_id, keeping the first occurrence
df_orders_deduplicated = df_orders_cleaned.dropDuplicates(["order_id"])
print("Jumlah baris sebelum deduplikasi orders:", df_orders_cleaned.count())
print("Jumlah baris sesudah deduplikasi orders:", df_orders_deduplicated.count())
print()

# deduplicate order_items by order_id and product_id, keeping the first occurrence
df_order_items_deduplicated = df_order_items_cleaned.dropDuplicates(["order_id", "product_id"])
print("Jumlah baris sebelum deduplikasi order_items:", df_order_items_cleaned.count())
print("Jumlah baris sesudah deduplikasi order_items:", df_order_items_deduplicated.count())
print()

# deduplicate products by product_id, keeping the first occurrence
df_products_deduplicated = df_products_standardized.dropDuplicates(["product_id"])
print("Jumlah baris sebelum deduplikasi products:", df_products_standardized.count())
print("Jumlah baris sesudah deduplikasi products:", df_products_deduplicated.count())
print()

# deduplicate users by user_id, keeping the first occurrence
df_users_deduplicated = df_users_standardized.dropDuplicates(["user_id"])
print("Jumlah baris sebelum deduplikasi users:", df_users_standardized.count())
print("Jumlah baris sesudah deduplikasi users:", df_users_deduplicated.count())

Jumlah baris sebelum deduplikasi orders: 16
Jumlah baris sesudah deduplikasi orders: 15

Jumlah baris sebelum deduplikasi order_items: 18
Jumlah baris sesudah deduplikasi order_items: 17

Jumlah baris sebelum deduplikasi products: 9
Jumlah baris sesudah deduplikasi products: 8

Jumlah baris sebelum deduplikasi users: 13
Jumlah baris sesudah deduplikasi users: 12


# SOAL 3 — Data Enrichment & Feature Engineering

## Perintah 1:
Di order_items, buat kolom turunan:
1. total_item_value = quantity * price.

In [21]:
# total_item_value = quantity * price
df_order_items_enriched = df_order_items_deduplicated.withColumn("total_item_value", F.col("quantity") * F.col("price"))
df_order_items_enriched.show(5)

+--------+----------+--------+---------+----------------+
|order_id|product_id|quantity|    price|total_item_value|
+--------+----------+--------+---------+----------------+
|    O015|      P007|       2| 450000.0|        900000.0|
|    O011|      P001|       1|3500000.0|       3500000.0|
|    O007|      P003|      -1| 150000.0|       -150000.0|
|    O009|      P007|       1| 450000.0|        450000.0|
|    O008|      P004|       1| -50000.0|        -50000.0|
+--------+----------+--------+---------+----------------+
only showing top 5 rows


## Perintah 2:
Di orders, buat kolom kategorisasi menggunakan when/otherwise:
1. order_value_flag dengan aturan:
    - high_value jika total_amount > 500000, selain itu normal.

In [22]:
# order_value_flag 
df_orders_enriched = df_orders_deduplicated.withColumn("order_value_flag", F.when(F.col("total_amount") > 500000, "high_value").otherwise("normal"))
df_orders_enriched.show(5)

+--------+-------+----------+------------+----------------+
|order_id|user_id|order_date|total_amount|order_value_flag|
+--------+-------+----------+------------+----------------+
|    O001|   U001|2023-10-01|   3500000.0|      high_value|
|    O002|   U001|2023-10-15|    400000.0|          normal|
|    O003|   U002|2023-10-02|   8500000.0|      high_value|
|    O004|   U003|2023-10-03|    150000.0|          normal|
|    O005|   U004|2023-10-04|    300000.0|          normal|
+--------+-------+----------+------------+----------------+
only showing top 5 rows


## Perintah 3:
Buat kolom segmentasi user di users:
1. user_tenure_days = datediff(order_date_max_per_user, signup_date) (Hint: butuh join/agg kecil untuk order_date_max_per_user).
2. tenure_segment = new (<=30 hari), mature (31–180), loyal (>180).

In [23]:
# order_date_max_per_user
df_max_order_date = df_orders_enriched \
    .groupBy("user_id")\
        .agg(
            F.max("order_date").alias("order_date_max_per_user")
        )
df_max_order_date.show(5)

+-------+-----------------------+
|user_id|order_date_max_per_user|
+-------+-----------------------+
|   U007|             2023-10-07|
|   U004|             2023-10-04|
|   U011|             2023-10-22|
|   U009|             2023-10-09|
|   U005|             2023-10-05|
+-------+-----------------------+
only showing top 5 rows


In [24]:
# Join df_max_order_date with df_users_deduplicated to get the most recent order for each user
df_users_enriched = df_users_deduplicated \
    .join(df_max_order_date, on="user_id", how="left")

In [25]:
# user_tenure_days
df_users_enriched = df_users_enriched \
    .withColumn(
        "user_tenure_days", 
        F.datediff(
            F.col("order_date_max_per_user"), 
            F.col("signup_date")
        )
    )

df_users_enriched.show(5)

+-------+-------------+--------+-----------+-----------------------+----------------+
|user_id|         name|    city|signup_date|order_date_max_per_user|user_tenure_days|
+-------+-------------+--------+-----------+-----------------------+----------------+
|   U001| Andi Pratama| jakarta| 2023-01-15|             2023-10-20|             278|
|   U002| Budi Santoso|surabaya| 2023-03-20|             2023-10-21|             215|
|   U003|   Citra Dewi| bandung| 2023-06-10|             2023-10-03|             115|
|   U004|Dimas Anggara| jakarta| 2023-07-01|             2023-10-04|              95|
|   U005|    Eka Putri|   medan| 2023-08-15|             2023-10-05|              51|
+-------+-------------+--------+-----------+-----------------------+----------------+
only showing top 5 rows


In [26]:
# tenure_segment
df_users_enriched = df_users_enriched \
    .withColumn(
        "tenure_segment",
        F.when(F.col("user_tenure_days").isNull(), "unknown") 
         .when(F.col("user_tenure_days") <= 30, "new") 
         .when((F.col("user_tenure_days") <= 180), "mature") \
         .otherwise("loyal")
    )

df_users_enriched.show(5)

+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
|user_id|         name|    city|signup_date|order_date_max_per_user|user_tenure_days|tenure_segment|
+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
|   U001| Andi Pratama| jakarta| 2023-01-15|             2023-10-20|             278|         loyal|
|   U002| Budi Santoso|surabaya| 2023-03-20|             2023-10-21|             215|         loyal|
|   U003|   Citra Dewi| bandung| 2023-06-10|             2023-10-03|             115|        mature|
|   U004|Dimas Anggara| jakarta| 2023-07-01|             2023-10-04|              95|        mature|
|   U005|    Eka Putri|   medan| 2023-08-15|             2023-10-05|              51|        mature|
+-------+-------------+--------+-----------+-----------------------+----------------+--------------+
only showing top 5 rows


# SOAL 4 — Join & Aggregation (Business Metrics)

## Perintah 1:
Bangun DataFrame “fact” hasil join:
<br>orders ⟷ order_items ⟷ products ⟷ users dengan key:
1. orders ↔ order_items: order_id
2. order_items ↔ products: product_id
3. orders ↔ users: user_id


In [27]:
# df_fact_orders
df_fact_orders = df_orders_enriched \
    .join(df_order_items_enriched, on="order_id", how="left") \
    .join(df_products_deduplicated, on="product_id", how="left") \
    .join(df_users_enriched, on="user_id", how="left")

df_fact_orders.show(5)
df_fact_orders.printSchema()

+-------+----------+--------+----------+------------+----------------+--------+---------+----------------+-----------------+------------+---------+-------------+--------+-----------+-----------------------+----------------+--------------+
|user_id|product_id|order_id|order_date|total_amount|order_value_flag|quantity|    price|total_item_value|     product_name|    category|    price|         name|    city|signup_date|order_date_max_per_user|user_tenure_days|tenure_segment|
+-------+----------+--------+----------+------------+----------------+--------+---------+----------------+-----------------+------------+---------+-------------+--------+-----------+-----------------------+----------------+--------------+
|   U001|      P001|    O001|2023-10-01|   3500000.0|      high_value|       1|3500000.0|       3500000.0|     Smartphone X|  elektronik|3500000.0| Andi Pratama| jakarta| 2023-01-15|             2023-10-20|             278|         loyal|
|   U001|      P008|    O002|2023-10-15|    

# Perintah 2:
Hitung metrik per <b>category per day</b>:
1. daily_gmv_category = sum(total_item_value)
2. daily_orders_category = countDistinct(order_id)
3. daily_items_sold = sum(quantity)
<br>Urutkan per hari dan GMV terbesar.

In [28]:
# daily_gmv_category
df_daily_category = df_fact_orders \
    .groupBy("order_date", "category") \
    .agg(
        F.sum("total_item_value").alias("daily_gmv"),
        F.countDistinct("order_id").alias("daily_orders_category"),
        F.sum("quantity").alias("daily_items_sold")

    ) \
    .orderBy(
        F.col("order_date").asc(),
        F.col("category").desc()
    )

df_daily_category.show(20)

+----------+------------+---------+---------------------+----------------+
|order_date|    category|daily_gmv|daily_orders_category|daily_items_sold|
+----------+------------+---------+---------------------+----------------+
|2023-10-01|  elektronik|3500000.0|                    1|               1|
|2023-10-02|  elektronik|8500000.0|                    1|               1|
|2023-10-03|     fashion| 150000.0|                    1|               1|
|2023-10-04|rumah tangga| 300000.0|                    1|               1|
|2023-10-05|rumah tangga|1200000.0|                    1|               1|
|2023-10-06|     fashion|-150000.0|                    1|              -1|
|2023-10-07|     fashion| -50000.0|                    1|               1|
|2023-10-08|     fashion| 700000.0|                    1|               2|
|2023-10-09|     fashion| 450000.0|                    1|               1|
|2023-10-10|  elektronik|3700000.0|                    1|               2|
|2023-10-15|  elektronik|

## Perintah 3:
Hitung metrik per city:
1. total GMV kota,
2. AOV kota (avg total_amount),
3. jumlah customer unik kota,
4. top 3 kategori kontribusi terbesar per kota (boleh pakai window rank).

In [29]:
# Metrik 1-3: Total GMV, AOV, dan Unique Customers per City
df_city_metrics = (
    df_fact_orders
    .groupBy("city")
    .agg(
        F.sum("total_item_value").alias("total_gmv_city"),
        F.avg("total_amount").alias("aov_city"),
        F.countDistinct("user_id").alias("unique_customers_city")
    )
)

df_city_metrics.show(truncate=False)

+----------+--------------+---------+---------------------+
|city      |total_gmv_city|aov_city |unique_customers_city|
+----------+--------------+---------+---------------------+
|semarang  |-50000.0      |0.0      |1                    |
|jakarta   |4850000.0     |970000.0 |3                    |
|surabaya  |9450000.0     |2537500.0|2                    |
|yogyakarta|600000.0      |600000.0 |1                    |
|medan     |1200000.0     |1200000.0|1                    |
|bandung   |3700000.0     |1875000.0|3                    |
|makassar  |900000.0      |900000.0 |1                    |
+----------+--------------+---------+---------------------+



In [30]:
# Metrik 4: Top 3 Categories by GMV per City
# Hitung GMV setiap kategori di setiap kota
df_city_category_gmv = (
    df_fact_orders
    .groupBy("city", "category")
    .agg(
        F.sum("total_item_value").alias("category_gmv")
    )
)

# Ranking kategori berdasarkan GMV dalam setiap kota
window_city = Window.partitionBy("city").orderBy(
    F.col("category_gmv").desc()
)

# Ambil 3 kategori teratas berdasarkan GMV untuk setiap kota
df_top3_category_city = (
    df_city_category_gmv
    .withColumn("category_rank", F.dense_rank().over(window_city))
    .filter(F.col("category_rank") <= 3)
    .orderBy("city", "category_rank")
)

df_top3_category_city.show(truncate=False)

+----------+------------+------------+-------------+
|city      |category    |category_gmv|category_rank|
+----------+------------+------------+-------------+
|bandung   |elektronik  |3700000.0   |1            |
|bandung   |fashion     |0.0         |2            |
|jakarta   |elektronik  |4100000.0   |1            |
|jakarta   |fashion     |450000.0    |2            |
|jakarta   |rumah tangga|300000.0    |3            |
|makassar  |fashion     |900000.0    |1            |
|medan     |rumah tangga|1200000.0   |1            |
|semarang  |fashion     |-50000.0    |1            |
|surabaya  |elektronik  |8500000.0   |1            |
|surabaya  |fashion     |950000.0    |2            |
|yogyakarta|rumah tangga|600000.0    |1            |
+----------+------------+------------+-------------+



In [31]:
# Combine city metrics with top 3 categories per city
df_city_summary = (
    df_city_metrics
    .join(df_top3_category_city, on="city", how="left")
    .orderBy("city", "category_rank")
)

df_city_summary.show(truncate=False)

+----------+--------------+---------+---------------------+------------+------------+-------------+
|city      |total_gmv_city|aov_city |unique_customers_city|category    |category_gmv|category_rank|
+----------+--------------+---------+---------------------+------------+------------+-------------+
|bandung   |3700000.0     |1875000.0|3                    |elektronik  |3700000.0   |1            |
|bandung   |3700000.0     |1875000.0|3                    |fashion     |0.0         |2            |
|jakarta   |4850000.0     |970000.0 |3                    |elektronik  |4100000.0   |1            |
|jakarta   |4850000.0     |970000.0 |3                    |fashion     |450000.0    |2            |
|jakarta   |4850000.0     |970000.0 |3                    |rumah tangga|300000.0    |3            |
|makassar  |900000.0      |900000.0 |1                    |fashion     |900000.0    |1            |
|medan     |1200000.0     |1200000.0|1                    |rumah tangga|1200000.0   |1            |


# SOAL 5 — Window Analytics & Insight Final

## Perintah 1:
Gunakan window function per user untuk membuat kolom:
1. user_total_spending (sum total_amount per user),
2. user_total_orders (count order per user),
3. user_avg_order_value (avg total_amount per user),
4. user_rank global berdasarkan user_total_spending (dense_rank).

In [32]:
# 1. Ambil satu baris per order
df_orders_unique = (
    df_fact_orders
    .select("user_id", "order_id", "total_amount")
    .dropDuplicates(["order_id"])
)

# 2. Agregasi metrik per user
df_user_agg = (
    df_orders_unique
    .groupBy("user_id")
    .agg(
        F.sum("total_amount").alias("user_total_spending"),
        F.countDistinct("order_id").alias("user_total_orders"),
        F.avg("total_amount").alias("user_avg_order_value")
    )
)

# 3. Ranking global berdasarkan spending terbesar
window_rank = Window.orderBy(
    F.col("user_total_spending").desc()
)

df_user_metrics = (
    df_user_agg
    .withColumn(
        "user_rank",
        F.dense_rank().over(window_rank)
    )
)

# 4. Tampilkan hasil
df_user_metrics.orderBy("user_rank").show(20, truncate=False)

+-------+-------------------+-----------------+--------------------+---------+
|user_id|user_total_spending|user_total_orders|user_avg_order_value|user_rank|
+-------+-------------------+-----------------+--------------------+---------+
|U002   |8750000.0          |2                |4375000.0           |1        |
|U001   |4100000.0          |3                |1366666.6666666667  |2        |
|U010   |3700000.0          |1                |3700000.0           |3        |
|U005   |1200000.0          |1                |1200000.0           |4        |
|U012   |900000.0           |1                |900000.0            |5        |
|U008   |700000.0           |1                |700000.0            |6        |
|U011   |600000.0           |1                |600000.0            |7        |
|U009   |450000.0           |1                |450000.0            |8        |
|U004   |300000.0           |1                |300000.0            |9        |
|U003   |150000.0           |1                |15000

## Perintah 2:
Buat analisis cumulative GMV harian:
1. gmv_daily,
2. cumulative_gmv (sum gmv_daily over order by date).

In [33]:
# 1. Hitung GMV harian
df_gmv_daily = (
    df_fact_orders
    .groupBy("order_date")
    .agg(
        F.sum("total_item_value").alias("gmv_daily")
    )
)

# 2. Window untuk cumulative GMV
window_cumulative = (
    Window
    .orderBy("order_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# 3. Hitung cumulative GMV
df_cumulative_gmv = (
    df_gmv_daily
    .withColumn(
        "cumulative_gmv",
        F.sum("gmv_daily").over(window_cumulative)
    )
    .orderBy("order_date")
)

# 4. Tampilkan hasil
df_cumulative_gmv.show(30, truncate=False)

+----------+---------+--------------+
|order_date|gmv_daily|cumulative_gmv|
+----------+---------+--------------+
|2023-10-01|3500000.0|3500000.0     |
|2023-10-02|8500000.0|1.2E7         |
|2023-10-03|150000.0 |1.215E7       |
|2023-10-04|300000.0 |1.245E7       |
|2023-10-05|1200000.0|1.365E7       |
|2023-10-06|-150000.0|1.35E7        |
|2023-10-07|-50000.0 |1.345E7       |
|2023-10-08|700000.0 |1.415E7       |
|2023-10-09|450000.0 |1.46E7        |
|2023-10-10|3700000.0|1.83E7        |
|2023-10-15|400000.0 |1.87E7        |
|2023-10-20|200000.0 |1.89E7        |
|2023-10-21|250000.0 |1.915E7       |
|2023-10-22|600000.0 |1.975E7       |
|2023-10-23|900000.0 |2.065E7       |
+----------+---------+--------------+



## Perintah 3:
Ambil Top 10 customers (rank 1–10) dan tampilkan:
user_id, name, city, user_total_orders, user_total_spending, user_avg_order_value.
Tambahkan interpretasi singkat 4–6 kalimat: pola apa yang kamu lihat? (misal kota dominan, AOV tinggi tapi order sedikit, dll).

In [34]:
# Ambil informasi user
df_users_info = df_users_deduplicated.select(
    "user_id",
    "name",
    "city"
)

# Join metrik customer dengan informasi user
df_top10_customers = (
    df_user_metrics
    .filter(F.col("user_rank").between(1, 10))
    .join(df_users_info, on="user_id", how="left")
    .select(
        "user_id",
        "name",
        "city",
        "user_total_orders",
        "user_total_spending",
        "user_avg_order_value"
    )
    .orderBy("user_total_spending", ascending=False)
)

df_top10_customers.show(10, truncate=False)

+-------+-------------+----------+-----------------+-------------------+--------------------+
|user_id|name         |city      |user_total_orders|user_total_spending|user_avg_order_value|
+-------+-------------+----------+-----------------+-------------------+--------------------+
|U002   |Budi Santoso |surabaya  |2                |8750000.0          |4375000.0           |
|U001   |Andi Pratama |jakarta   |3                |4100000.0          |1366666.6666666667  |
|U010   |Joko Anwar   |bandung   |1                |3700000.0          |3700000.0           |
|U005   |Eka Putri    |medan     |1                |1200000.0          |1200000.0           |
|U012   |Lukman Hakim |makassar  |1                |900000.0           |900000.0            |
|U008   |Hadi Wijaya  |surabaya  |1                |700000.0           |700000.0            |
|U011   |Kiki Amelia  |yogyakarta|1                |600000.0           |600000.0            |
|U009   |Indah Sari   |jakarta   |1                |450000.0

Hasil Analisis
- Customer dengan spending tertinggi: Budi Santoso dari Surabaya memiliki total belanja Rp8.750.000 dari 2 order, dengan rata-rata Rp4.375.000 per order.
- Customer dengan order terbanyak: Andi Pratama dari Jakarta melakukan 3 order dengan total belanja Rp4.100.000.
- Kota yang paling dominan: Jakarta memiliki 3 customer dalam Top 10, sedangkan Surabaya memiliki 2 customer.
- AOV tinggi, tetapi order sedikit: Joko Anwar dari Bandung memiliki rata-rata belanja Rp3.700.000, tetapi baru melakukan 1 order.
- Peluang meningkatkan penjualan: Customer yang baru melakukan 1 order bisa ditargetkan dengan promosi agar melakukan pembelian kembali.
- Strategi retensi: Pertahankan customer dengan nilai belanja tinggi seperti Budi dan Andi melalui program loyalitas atau penawaran khusus.